# Race exploration: Bahrain 2022

In this notebook I explore one race before cleaning anything. I want to know
what the data contains, what is missing and what looks odd, so that the
cleaning rules I write later are based on what I observed and not on guesses.
The code that loads the data lives in `src/f1_telemetry`; the notebook only
calls it. To explore another race, I only change the three constants in the
loading cell.

## 0. Setup

When I edit a module of the package, Jupyter reloads it automatically, so I do
not have to restart the kernel.

In [1]:
%load_ext autoreload
%autoreload 2

The package lives in `src/`. This cell finds the project root (the folder with
`pyproject.toml`) and adds `src/` to the import path, so the notebook runs on a
fresh clone after only `pip install -r requirements.txt`.

In [2]:
import sys
from pathlib import Path

# The kernel may start in notebooks/ or in the project root: look upwards
# for pyproject.toml, which only exists in the root.
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)

# Let Python find the package, which lives in src/ (no pip install needed)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

## 1. Load the race

`YEAR`, `GRAND_PRIX` and `SESSION_TYPE` choose the session. `load_session`
looks the race up in the season calendar instead of letting FastF1 guess the
name. I keep FastF1 warnings visible because they report data problems: here it
says that driver 16 (LEC, the winner) completed the race distance 0.05 s before
the recorded end of the session.
I get two tables: 1125 laps with 31 columns, and the results of 20 drivers with
22 columns.

In [3]:
import fastf1
import pandas as pd

from f1_telemetry.data_loader import (
    LAP_DATETIME_COLUMNS,
    LAP_TIMEDELTA_COLUMNS,
    get_laps,
    get_results,
    load_dataframe,
    load_session,
    save_dataframe,
    setup_cache,
    snapshot_path,
)

# hide the INFO lines, keep warnings (they report data problems)
fastf1.set_log_level("WARNING")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

YEAR = 2022
GRAND_PRIX = "Bahrain"
SESSION_TYPE = "R"

setup_cache()
session = load_session(YEAR, GRAND_PRIX, SESSION_TYPE)
laps = get_laps(session)
results = get_results(session)

print(session.event["EventName"], YEAR, "-", session.name)
print("laps:", laps.shape, "| results:", results.shape)

core        WARNING 	Driver 16 completed the race distance 00:00.050000 before the recorded end of the session.


Bahrain Grand Prix 2022 - Race
laps: (1125, 31) | results: (20, 22)


## 2. Official results

I keep the columns useful for a strategy analysis: grid and finishing position,
status, laps completed and points. `Time` is the winner's race time and, for
the others, a value that grows with position (it looks like the gap to the
winner); it is empty for the three retirements. LEC won from pole. 17 drivers
finished 57 laps; PER, VER and GAS retired. GAS has `ClassifiedPosition` "R"
(not classified) although his `Position` is 20.

In [4]:
wanted = [
    "DriverNumber",
    "Abbreviation",
    "TeamName",
    "GridPosition",
    "Position",
    "ClassifiedPosition",
    "Status",
    "Laps",
    "Points",
    "Time",
]
results[wanted]

,DriverNumber,Abbreviation,TeamName,GridPosition,Position,ClassifiedPosition,Status,Laps,Points,Time
0,16,LEC,Ferrari,1.0,1.0,1,Finished,57.0,26.0,0 days 01:37:33.584000
1,55,SAI,Ferrari,3.0,2.0,2,Finished,57.0,18.0,0 days 00:00:05.598000
2,44,HAM,Mercedes,5.0,3.0,3,Finished,57.0,15.0,0 days 00:00:09.675000
3,63,RUS,Mercedes,9.0,4.0,4,Finished,57.0,12.0,0 days 00:00:11.211000
4,20,MAG,Haas F1 Team,7.0,5.0,5,Finished,57.0,10.0,0 days 00:00:14.754000
5,77,BOT,Alfa Romeo,6.0,6.0,6,Finished,57.0,8.0,0 days 00:00:16.119000
6,31,OCO,Alpine,11.0,7.0,7,Finished,57.0,6.0,0 days 00:00:19.423000
7,22,TSU,AlphaTauri,16.0,8.0,8,Finished,57.0,4.0,0 days 00:00:20.386000
8,14,ALO,Alpine,8.0,9.0,9,Finished,57.0,2.0,0 days 00:00:22.390000
9,24,ZHO,Alfa Romeo,15.0,10.0,10,Finished,57.0,1.0,0 days 00:00:23.064000


These are all the 22 columns of the results table. The others are names,
colours, a headshot link and the qualifying times `Q1`, `Q2`, `Q3`; I do not
need them for a race.

In [5]:
results.columns.tolist()

['DriverNumber',
 'BroadcastName',
 'Abbreviation',
 'DriverId',
 'TeamName',
 'TeamColor',
 'TeamId',
 'FirstName',
 'LastName',
 'FullName',
 'HeadshotUrl',
 'CountryCode',
 'Position',
 'ClassifiedPosition',
 'GridPosition',
 'Q1',
 'Q2',
 'Q3',
 'Time',
 'Status',
 'Points',
 'Laps']

## 3. The lap table

One row is one lap of one driver. The first five laps of VER show what a normal
lap looks like. On lap 1 `Sector1Time` is empty although `LapTime` exists, and
`LapStartDate` is empty on every row. His tyres are SOFT with `TyreLife` 4 and
`FreshTyre` False: he started the race on a used set.

In [6]:
laps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 01:04:15.340000,VER,1,0 days 00:01:40.236000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:42.325000,0 days 00:00:24.389000,NaT,0 days 01:03:51.046000,0 days 01:04:15.427000,230.0,254.0,274.0,250.0,False,SOFT,4.0,False,Red Bull Racing,0 days 01:02:34.872000,NaT,1,2.0,False,,False,False
1,0 days 01:05:53.220000,VER,1,0 days 00:01:37.880000,2.0,1.0,NaT,NaT,0 days 00:00:31.285000,0 days 00:00:42.269000,0 days 00:00:24.326000,0 days 01:04:46.662000,0 days 01:05:28.931000,0 days 01:05:53.257000,230.0,252.0,276.0,296.0,True,SOFT,5.0,False,Red Bull Racing,0 days 01:04:15.340000,NaT,1,2.0,False,,False,True
2,0 days 01:07:31.577000,VER,1,0 days 00:01:38.357000,3.0,1.0,NaT,NaT,0 days 00:00:31.499000,0 days 00:00:42.474000,0 days 00:00:24.384000,0 days 01:06:24.756000,0 days 01:07:07.230000,0 days 01:07:31.614000,229.0,254.0,276.0,294.0,False,SOFT,6.0,False,Red Bull Racing,0 days 01:05:53.220000,NaT,1,2.0,False,,False,True
3,0 days 01:09:10.143000,VER,1,0 days 00:01:38.566000,4.0,1.0,NaT,NaT,0 days 00:00:31.342000,0 days 00:00:42.674000,0 days 00:00:24.550000,0 days 01:08:02.956000,0 days 01:08:45.630000,0 days 01:09:10.180000,231.0,250.0,276.0,297.0,False,SOFT,7.0,False,Red Bull Racing,0 days 01:07:31.577000,NaT,1,2.0,False,,False,True
4,0 days 01:10:49.020000,VER,1,0 days 00:01:38.877000,5.0,1.0,NaT,NaT,0 days 00:00:31.498000,0 days 00:00:42.854000,0 days 00:00:24.525000,0 days 01:09:41.678000,0 days 01:10:24.532000,0 days 01:10:49.057000,229.0,256.0,276.0,293.0,False,SOFT,8.0,False,Red Bull Racing,0 days 01:09:10.143000,NaT,1,2.0,False,,False,True


The 31 columns, in groups: timing (`Time`, `LapTime`, the sector times),
pit stops and stints (`Stint`, `PitInTime`, `PitOutTime`), tyres (`Compound`,
`TyreLife`, `FreshTyre`), speeds (`SpeedI1`, `SpeedI2`, `SpeedFL`, `SpeedST`),
and context about the lap (`Position`, `TrackStatus`, `Deleted`,
`DeletedReason`, `FastF1Generated`, `IsAccurate`).

In [7]:
laps.columns.tolist()

['Time',
 'Driver',
 'DriverNumber',
 'LapTime',
 'LapNumber',
 'Stint',
 'PitOutTime',
 'PitInTime',
 'Sector1Time',
 'Sector2Time',
 'Sector3Time',
 'Sector1SessionTime',
 'Sector2SessionTime',
 'Sector3SessionTime',
 'SpeedI1',
 'SpeedI2',
 'SpeedFL',
 'SpeedST',
 'IsPersonalBest',
 'Compound',
 'TyreLife',
 'FreshTyre',
 'Team',
 'LapStartTime',
 'LapStartDate',
 'TrackStatus',
 'Position',
 'Deleted',
 'DeletedReason',
 'FastF1Generated',
 'IsAccurate']

`info()` gives the type of each column and how many values are present.
- Times are timedeltas; `LapNumber` and `Stint` are floats.
- `LapTime` has 1118 values out of 1125: 7 laps have no lap time.
- `PitOutTime` has 58 values and `PitInTime` 59.
- `Sector1Time` has 1103 values and `SpeedFL` 1064: some laps lack a sector or
  speed measurement.
- `LapStartDate` has no values at all in this race.

In [8]:
laps.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1125 entries, 0 to 1124
Data columns (total 31 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   Time                1125 non-null   timedelta64[ns]
 1   Driver              1125 non-null   object         
 2   DriverNumber        1125 non-null   object         
 3   LapTime             1118 non-null   timedelta64[ns]
 4   LapNumber           1125 non-null   float64        
 5   Stint               1125 non-null   float64        
 6   PitOutTime          58 non-null     timedelta64[ns]
 7   PitInTime           59 non-null     timedelta64[ns]
 8   Sector1Time         1103 non-null   timedelta64[ns]
 9   Sector2Time         1123 non-null   timedelta64[ns]
 10  Sector3Time         1123 non-null   timedelta64[ns]
 11  Sector1SessionTime  1101 non-null   timedelta64[ns]
 12  Sector2SessionTime  1123 non-null   timedelta64[ns]
 13  Sector3SessionTime  1123 non-null

## 4. Retirements

Three drivers did not finish: PER (56 laps, fuel pressure), VER (54 laps, fuel
pressure) and GAS (44 laps, power unit).

In [9]:
results.loc[results["Status"] != "Finished", ["Abbreviation", "Laps", "Status"]].head(5)

,Abbreviation,Laps,Status
17,PER,56.0,Fuel pressure
18,VER,54.0,Fuel pressure
19,GAS,44.0,Power Unit


`Position` is the running position at the end of each lap. 99.8% of the laps
have it; I look at the missing ones below.

In [10]:
print("Share of laps with a Position:", round(laps["Position"].notna().mean(), 3))
laps[["Driver", "LapNumber", "Position"]].head(10)

Share of laps with a Position: 0.998


,Driver,LapNumber,Position
0,VER,1.0,2.0
1,VER,2.0,2.0
2,VER,3.0,2.0
3,VER,4.0,2.0
4,VER,5.0,2.0
5,VER,6.0,2.0
6,VER,7.0,2.0
7,VER,8.0,2.0
8,VER,9.0,2.0
9,VER,10.0,2.0


## 5. Stints and tyres

The highest `Stint` number of each driver: 18 drivers have 4 stints (3 pit
stops) and MSC and GAS have 3.

In [11]:
laps.groupby("Driver")["Stint"].max().sort_values(ascending=False)

Driver
ALB    4.0
ALO    4.0
VER    4.0
TSU    4.0
STR    4.0
SAI    4.0
RUS    4.0
RIC    4.0
PER    4.0
OCO    4.0
NOR    4.0
MAG    4.0
LEC    4.0
LAT    4.0
HUL    4.0
HAM    4.0
BOT    4.0
ZHO    4.0
MSC    3.0
GAS    3.0
Name: Stint, dtype: float64

Only three compounds appear: SOFT (675 laps), MEDIUM (345) and HARD (105), and
there is no missing value in `Compound`. `TyreLife` goes from 1 to 24 laps,
with a median of 9.

In [12]:
print(laps["Compound"].value_counts(dropna=False))
laps["TyreLife"].describe()

Compound
SOFT      675
MEDIUM    345
HARD      105
Name: count, dtype: int64


count    1125.000000
mean        9.128000
std         4.975703
min         1.000000
25%         5.000000
50%         9.000000
75%        13.000000
max        24.000000
Name: TyreLife, dtype: float64

## 6. Pit stops against stints

A stint starts after each pit stop, so I expect one more stint than out-laps
(laps that leave the pits). I count in-laps (`PitInTime`) and out-laps
(`PitOutTime`) per driver. This holds for every driver; the only one with an
extra in-lap is VER (4 in-laps, 3 out-laps): his last lap is an in-lap that is
not followed by an out-lap, which fits his retirement. In total there are 58
out-laps and 59 in-laps.

In [13]:
pit_check = pd.DataFrame(
    {
        "stints": laps.groupby("Driver")["Stint"].max(),
        "in_laps": laps.groupby("Driver")["PitInTime"].count(),
        "out_laps": laps.groupby("Driver")["PitOutTime"].count(),
    }
)
pit_check["in_minus_out"] = pit_check["in_laps"] - pit_check["out_laps"]
pit_check.sort_values("in_minus_out", ascending=False)

,stints,in_laps,out_laps,in_minus_out
Driver,,,,
VER,4.0,4,3,1
ALB,4.0,3,3,0
ALO,4.0,3,3,0
TSU,4.0,3,3,0
STR,4.0,3,3,0
SAI,4.0,3,3,0
RUS,4.0,3,3,0
RIC,4.0,3,3,0
PER,4.0,3,3,0


## 7. Laps without a position

Two laps have no `Position`: GAS lap 45 and PER lap 57. Both also have no
`LapTime`. The results say GAS completed 44 laps and PER 56, so these are the
first lap after the last completed lap, started but never finished.

In [14]:
laps.loc[laps["Position"].isna(), ["Driver", "LapNumber", "LapTime", "Position"]]

,Driver,LapNumber,LapTime,Position
98,GAS,45.0,NaT,NaN
155,PER,57.0,NaT,NaN


## 8. A stint table

I group the laps by driver and stint and keep the compound, the first and last
lap, and the tyre age at the start and at the end (first 12 rows shown). Two
things stand out: some stints start on used tyres (`TyreLifeStart` of 3 or 4,
for example ALO and BOT in stint 1), and a driver can use the same compound
twice in a row (ALB and BOT: MEDIUM, then MEDIUM). Later this code will move
into the package.

In [15]:
stints = laps.groupby(["Driver", "Stint"]).agg(
    Compound=("Compound", "first"),
    FirstLap=("LapNumber", "min"),
    LastLap=("LapNumber", "max"),
    TyreLifeStart=("TyreLife", "first"),
    TyreLifeEnd=("TyreLife", "last"),
)
stints.head(12)

Compound  FirstLap  LastLap  TyreLifeStart  TyreLifeEnd
Driver Stint                                                        
ALB    1.0       SOFT       1.0     13.0            1.0         13.0
       2.0     MEDIUM      14.0     35.0            1.0         22.0
       3.0     MEDIUM      36.0     46.0            3.0         13.0
       4.0       SOFT      47.0     57.0            4.0         14.0
ALO    1.0       SOFT       1.0     11.0            4.0         14.0
       2.0     MEDIUM      12.0     25.0            1.0         14.0
       3.0       HARD      26.0     42.0            1.0         17.0
       4.0       SOFT      43.0     57.0            1.0         15.0
BOT    1.0       SOFT       1.0     14.0            4.0         17.0
       2.0     MEDIUM      15.0     36.0            1.0         22.0
       3.0     MEDIUM      37.0     45.0            1.0          9.0
       4.0       SOFT      46.0     57.0            3.0         14.0

The official laps completed, in finishing order. I use them in the checks
below to compare with the lap table.

In [16]:
results[["Abbreviation", "Position", "Laps", "Status"]].sort_values("Position")

,Abbreviation,Position,Laps,Status
0,LEC,1.0,57.0,Finished
1,SAI,2.0,57.0,Finished
2,HAM,3.0,57.0,Finished
3,RUS,4.0,57.0,Finished
4,MAG,5.0,57.0,Finished
5,BOT,6.0,57.0,Finished
6,OCO,7.0,57.0,Finished
7,TSU,8.0,57.0,Finished
8,ALO,9.0,57.0,Finished
9,ZHO,10.0,57.0,Finished


## 9. Consistency checks

**Check 1.** The only driver where in-laps and out-laps differ is VER, the
pit retirement described above.

In [17]:
# 1) Every driver's stints - 1 should equal out-laps, except the pit retirement
pit_check[pit_check["in_minus_out"] != 0]

,stints,in_laps,out_laps,in_minus_out
Driver,,,,
VER,4.0,4,3,1


**Check 2.** In each stint, the number of laps (`LastLap - FirstLap + 1`) should
equal the tyre-life count (`TyreLifeEnd - TyreLifeStart + 1`). The result is an
empty table, so no stint is missing a lap.

In [18]:
# 2) Stints where lap count and tyre-life count disagree (expect: empty)
stints["laps_in_stint"] = stints["LastLap"] - stints["FirstLap"] + 1
stints["tyre_laps"] = stints["TyreLifeEnd"] - stints["TyreLifeStart"] + 1
stints[stints["laps_in_stint"] != stints["tyre_laps"]]

,,Compound,FirstLap,LastLap,TyreLifeStart,TyreLifeEnd,laps_in_stint,tyre_laps
Driver,Stint,,,,,,,


**Check 3.** I compare the laps in the results with the highest `LapNumber` in
the lap table. They differ by one for PER (56 and 57) and GAS (44 and 45),
because the lap table also holds the unfinished lap. So `max(LapNumber)` is not
the number of laps completed; I take it from the results.

In [19]:
# 3) Laps in the results vs laps counted in the lap table
laps_done = laps.groupby("Driver")["LapNumber"].max().rename("LapsCompleted")
check = results.set_index("Abbreviation")[["Laps"]].join(laps_done)
check[check["Laps"] != check["LapsCompleted"]]

,Laps,LapsCompleted
Abbreviation,,
PER,56.0,57.0
GAS,44.0,45.0


## 10. Laps without a lap time

7 laps have no `LapTime`. Two are the unfinished laps already seen (GAS lap 45
and PER lap 57). The other five (VER lap 48, LEC laps 47 and 48, ZHO lap 46,
RUS lap 46) all have track status 4, which FastF1 documents as the safety car.
Three of them (LEC 47, ZHO 46, RUS 46) are out-laps, since they have a
`PitOutTime`. All seven laps have `IsAccurate` False.

In [20]:
# Laps without a LapTime: what are the 7?
laps.loc[
    laps["LapTime"].isna(),
    ["Driver", "LapNumber", "PitInTime", "PitOutTime", "TrackStatus", "IsAccurate"],
]

,Driver,LapNumber,PitInTime,PitOutTime,TrackStatus,IsAccurate
47,VER,48.0,NaT,NaT,4,False
98,GAS,45.0,NaT,NaT,164,False
155,PER,57.0,NaT,NaT,12,False
259,LEC,47.0,NaT,0 days 02:19:18.088000,4,False
260,LEC,48.0,NaT,NaT,4,False
543,ZHO,46.0,NaT,0 days 02:19:14.342000,4,False
1056,RUS,46.0,NaT,0 days 02:18:19.190000,4,False


## 11. Flags on the laps

Counting the True values: 4 laps are `Deleted` (deleted by the stewards, for
example for track limits), 2 are `FastF1Generated` (laps added by FastF1, with
limited information), and 893 of 1125 laps (79%) are `IsAccurate`. `IsAccurate`
only says that the start and end times of the lap are synced with the other
laps; it is not about the precision of the lap time. In the documentation, a
lap is not accurate if it is an in-lap or out-lap, was not under a green or
yellow flag, or is missing a sector time.

In [21]:
# How many laps are flagged deleted, generated or accurate?
laps[["Deleted", "FastF1Generated", "IsAccurate"]].sum()

Deleted              4
FastF1Generated      2
IsAccurate         893
dtype: int64

## 12. Track status

`TrackStatus` is a text with the code of every status that occurred during the
lap. FastF1 documents 1 = track clear, 2 = yellow flag, 4 = safety car,
5 = red flag, 6 = virtual safety car, 7 = virtual safety car ending. In this
race 958 laps (85%) are only "1". The other 167 laps contain a yellow flag, a
virtual safety car or a safety car (for example "41" is safety car, then track
clear). There is no red flag. A lap with any other digit than 1 is not
representative of racing pace, so my pace analysis keeps only the laps whose
status is exactly "1".

In [22]:
# Track status values, as text
laps["TrackStatus"].value_counts()

TrackStatus
1      958
4       76
12      38
41      19
164     16
2        7
16       4
64       4
21       3
Name: count, dtype: int64

Four laps were deleted by the stewards, all for "TRACK LIMITS AT TURN 4" (ALO
lap 41, TSU lap 25, NOR lap 2, HAM lap 34). They still have a `LapTime` and a
green track status, so the lap-time column alone would not exclude them: I have
to filter on `Deleted` explicitly.

In [23]:
# Which laps are deleted?
laps.loc[
    laps["Deleted"],
    ["Driver", "LapNumber", "LapTime", "DeletedReason", "TrackStatus"],
]

,Driver,LapNumber,LapTime,DeletedReason,TrackStatus
196,ALO,41.0,0 days 00:01:39.854000,TRACK LIMITS AT TURN 4,1
408,TSU,25.0,0 days 00:01:40.376000,TRACK LIMITS AT TURN 4,1
727,NOR,2.0,0 days 00:01:42.279000,TRACK LIMITS AT TURN 4,1
816,HAM,34.0,0 days 00:01:37.593000,TRACK LIMITS AT TURN 4,1


The two `FastF1Generated` laps are GAS lap 45 and PER lap 57, the unfinished
laps already seen: no `LapTime`, with track status 164 and 12.

In [24]:
# Which laps were added by FastF1?
laps.loc[laps["FastF1Generated"], ["Driver", "LapNumber", "LapTime", "TrackStatus"]]

,Driver,LapNumber,LapTime,TrackStatus
98,GAS,45.0,NaT,164
155,PER,57.0,NaT,12


All 958 green laps have a lap time. 858 are `IsAccurate` and 100 are not: 20
are lap 1 and 80 are in-laps or out-laps. The last line, which shows the other
inaccurate laps, returns an empty table, so nothing else makes a green lap
inaccurate in this race. My explicit rules (not lap 1, not an in-lap or
out-lap, status exactly "1") therefore select the same laps as `IsAccurate`
here; I keep `IsAccurate` as a cross-check.

In [25]:
# Green laps with a time that are not IsAccurate: why?
green = laps[(laps["TrackStatus"] == "1") & laps["LapTime"].notna()]
not_acc = green[~green["IsAccurate"]]
pit_lap = not_acc["PitInTime"].notna() | not_acc["PitOutTime"].notna()
print("green laps with a time:", len(green), "| IsAccurate:", green["IsAccurate"].sum())
print(
    "not accurate:",
    len(not_acc),
    "| lap 1:",
    (not_acc["LapNumber"] == 1).sum(),
    "| in or out lap:",
    pit_lap.sum(),
)
not_acc[(not_acc["LapNumber"] > 1) & ~pit_lap]

green laps with a time: 958 | IsAccurate: 858
not accurate: 100 | lap 1: 20 | in or out lap: 80


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate


## 13. A very different race: Monaco 2022

My cleaning rules must also work on a wet or interrupted race. I load Monaco
into separate variables and look at the same things: compounds, track status,
deleted laps and laps without a time.

In [26]:
monaco_session = load_session(2022, "Monaco", "R")
monaco = get_laps(monaco_session)
monaco_results = get_results(monaco_session)

print("laps:", monaco.shape)
print(monaco["Compound"].value_counts(dropna=False))
print(monaco["TrackStatus"].value_counts())
print(
    "laps with a red flag code:",
    monaco["TrackStatus"].str.contains("5", na=False).sum(),
)
print(
    "deleted:", monaco["Deleted"].sum(), "| no LapTime:", monaco["LapTime"].isna().sum()
)

core        WARNING 	Fixed incorrect tyre stint information for driver '11'
core        WARNING 	Fixed incorrect tyre stint information for driver '55'
core        WARNING 	Fixed incorrect tyre stint information for driver '1'
core        WARNING 	Fixed incorrect tyre stint information for driver '16'
core        WARNING 	Fixed incorrect tyre stint information for driver '63'
core        WARNING 	Fixed incorrect tyre stint information for driver '4'
core        WARNING 	Fixed incorrect tyre stint information for driver '14'
core        WARNING 	Fixed incorrect tyre stint information for driver '44'
core        WARNING 	Fixed incorrect tyre stint information for driver '77'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
core        WARNING 	Fixed incorrect tyre stint information for driver '10'
core        WARNING 	Fixed incorrect tyre stint information for driver '31'
core        WARNING 	Fixed incorrect tyre stint information for driver '3'
core        WARN

laps: (1179, 31)
Compound
HARD            393
MEDIUM          346
WET             283
INTERMEDIATE    115
None             35
SOFT              7
Name: count, dtype: int64
TrackStatus
1       910
12      149
4        36
14       18
451      18
64       17
126      16
21        8
124       2
41        2
264       1
26        1
1264      1
Name: count, dtype: int64
laps with a red flag code: 18
deleted: 14 | no LapTime: 26


**What Monaco shows.** The lap table has 1179 rows. FastF1 printed one warning
per driver, "Fixed incorrect tyre stint information", so it corrected the stint
data of this race itself.
- Compounds: HARD 393, MEDIUM 346, WET 283, INTERMEDIATE 115, SOFT 7, and 35
  laps with the text "None" (unknown compound). 398 laps (34%) are on WET or
  INTERMEDIATE tyres; in Bahrain there are none.
- Track status: only 910 laps (77%) are exactly "1". 18 laps contain code 5
  (red flag), always with codes 4 and 1 ("451"), and virtual safety car codes
  (6) also appear.
- 14 laps are deleted and 26 have no `LapTime`.

My rule "track status exactly 1" still works, but a third of this race is on
wet tyres on a changing track, so the lap times cannot be compared. I will call
a race wet or mixed when at least 10% of its laps are on WET or INTERMEDIATE
tyres. For such a race I skip the pace comparison and the tyre degradation, and
say so; the tyre strategy, the evolution of positions and the grid-versus-finish
analysis stay valid.

________________________________________

   ## 14. Saving snapshots

   I save the lap table and the results of both races as CSV files in
   `data/raw/`. A later run (the app, the tests, a fresh clone) can read them
   without downloading anything from FastF1. I read one back to check that the
   time columns return as timedeltas.

In [27]:
for race, tables in [
    ("Bahrain", {"laps": laps, "results": results}),
    ("Monaco", {"laps": monaco, "results": monaco_results}),
]:
    for kind, table in tables.items():
        path = save_dataframe(table, snapshot_path(2022, race, "R", kind))
        print(path.relative_to(PROJECT_ROOT), "-", len(table), "rows")

back = load_dataframe(
    snapshot_path(2022, "Bahrain", "R", "laps"),
    timedelta_columns=LAP_TIMEDELTA_COLUMNS,
    datetime_columns=LAP_DATETIME_COLUMNS,
)
print(back.shape == laps.shape, back["LapTime"].dtype, back["DriverNumber"].dtype)

data/raw/2022_bahrain_R_laps.csv - 1125 rows
data/raw/2022_bahrain_R_results.csv - 20 rows
data/raw/2022_monaco_R_laps.csv - 1179 rows
data/raw/2022_monaco_R_results.csv - 20 rows
True timedelta64[ns] object
